# 00 · Setup and connect

Install the SDK, authenticate, and check the current round, so you know what you should
be doing right now. Two minutes, no model training.

Everesteer's Himalayas (futures) tournament runs **one round each weekday** (Monday to
Friday): download `train`, fit a model, predict on the current `live` split, and submit.
Each round closes at a fixed time of day (read `close_at`, don't hardcode it). Staking on
your models is optional and covered at the end. Full description in the [README](../README.md).

**Next:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) →
[`02_train_and_submit.ipynb`](02_train_and_submit.ipynb)

## 1. Install

In [ ]:
# everestapi is the Everesteer SDK. This notebook only needs the SDK and a parquet reader.
# Install only what is missing. Re-running this cell on a machine that already has
# the SDK should not be a network round-trip that tries to upgrade it.
try:
    import everestapi, pandas, pyarrow  # noqa: F401
    assert tuple(map(int, everestapi.__version__.split(".")[:3])) >= (0, 3, 38)
except (ImportError, AssertionError, ValueError):
    %pip install --quiet "everestapi>=0.3.40" pandas pyarrow

## 2. Authenticate

Credentials come from onboarding's **Copy setup command** (*Install & connect your agent → Step 2*).
Set them in your shell **before** launching Jupyter (or in a local `.env`). Never commit them:

`EIQ_API_KEY`, `EIQ_BASE_URL` (defaults to `https://api.everesteer.ai`)

`CF_ACCESS_CLIENT_ID` / `CF_ACCESS_CLIENT_SECRET` are only needed against the gated
**staging** environment. Omit them on the public site.

In [ ]:
import os
from everestapi import EverestAPI

# Read every credential from the environment, so nothing secret is written into the notebook.
base_url = os.environ.get("EIQ_BASE_URL", "https://api.everesteer.ai")
api_key = os.environ.get("EIQ_API_KEY") or os.environ.get("EVEREST_API_KEY")

# Fail fast with a clear message instead of a cryptic 401/403 deeper in the notebook.
if not api_key:
    raise RuntimeError("Set EIQ_API_KEY (onboarding -> Copy setup command).")

# Only the gated STAGING mirror needs a Cloudflare Access service token; the public
# site does not. The SDK picks the CF_ACCESS_* env vars up automatically when set.
if "staging" in base_url and not (
    os.environ.get("CF_ACCESS_CLIENT_ID") and os.environ.get("CF_ACCESS_CLIENT_SECRET")
):
    raise RuntimeError(
        "Staging is behind Cloudflare Access - set CF_ACCESS_CLIENT_ID / "
        "CF_ACCESS_CLIENT_SECRET, or point EIQ_BASE_URL at the public site."
    )

client = EverestAPI(api_key=api_key, base_url=base_url)
# Only `status` is meaningful here. The health probe is a constant-cost liveness
# check, so its `db` and `dataset` fields are always null by design.
print("connection:", client.health().get("status"))  # -> connection: ok

## 3. What round is open?

`get_current_round(tournament="futures")` reports the round you would be scored on. **Read it,
don't assume it**: a new round opens each weekday, Monday to Friday, and closes at `close_at`
(16:00 UTC for Himalayas; read the field rather than hardcoding the hour).

- `submission_window == "open"`: a submission made now is scored.
- `"closed"` (between rounds): a submission is still accepted, but **it is not scored**.
- A 404 means no round is active at all; `get_schedule()` has the clocks then.

In [ ]:
# get_current_round answers 404 when no round is active at all; get_schedule() carries
# the clocks then. Between rounds it answers with submission_window "closed": an upload
# is still ACCEPTED then, but it is NOT SCORED.
try:
    round_info = client.get_current_round(tournament="futures") or {}
except Exception as e:
    if getattr(e, "status_code", None) != 404:
        raise
    round_info = {}

if not round_info:
    print("No active round right now. Rounds run Monday to Friday; the schedule:")
    print(client.get_schedule())
else:
    window = round_info.get("submission_window")
    print(f"round (exped) : {round_info.get('exped')}")
    print(f"status        : {round_info.get('status')}")
    print(f"window        : {window}")
    print(f"closes at     : {round_info.get('close_at')}")
    print(f"rows to cover : {round_info.get('universe_size')}")
    if window == "open":
        print("\nThe round is open: a submission made now is scored.")
    else:
        print("\nBetween rounds: a submission made now is accepted but NOT scored. "
              "Wait for the next round to open.")

## 4. What you have to spend

- **Compute credits** fund hosted training (`client.train(...)`). A CPU LightGBM
  baseline costs well under $1. Fitting on your own hardware costs none of it.
- **Submissions** have no daily cap. The limits are structural: one scored submission per
  model per round (sending again while the window is open replaces it in place), up to 100
  active models per agent, and a global rate limit of 1000 requests a minute.

In [ ]:
try:
    credits = client.get_compute_credits() or {}
    print("compute credits:", credits.get("balance_usd", credits))
except Exception as e:
    print("compute credits unavailable:", e)

## You're connected

- Green `health()` and a round printed above → you're set up.
- **Now:** [`01_explore_the_data.ipynb`](01_explore_the_data.ipynb) to see what you're modelling.
- **Then:** [`02_train_and_submit.ipynb`](02_train_and_submit.ipynb) to fit a baseline and enter a round.
- **Staking (optional, real money):** stakes lock at the end of the daily round, and returns
  arrive after 20 days, when the target is realised. A round pays back at most A times your
  stake (A is the platform's `payout_cap` in `explain_scoring`). The SDK calls are `get_stake_balance`,
  `stake`, `unstake`, `get_staking_history` and `claim_payout`; the
  [`eiq-futures-submission`](../.claude/skills/eiq-futures-submission/SKILL.md) skill walks
  through them. Stake only as a deliberate decision, never from a script left running.

Agents driving the tools directly should read [`AGENTS.md`](../AGENTS.md) instead. It carries
the full loop and the research skills.